In [119]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    MultiHeadAttention,
    Dense,
    LayerNormalization,
    Dropout,
    Input
)

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [120]:
# # Read txt file

# with open("english_telugu_pairs_15000.txt", "r", encoding="utf-8") as f:
#     content = f.read()

# # Convert text into Python list
# data = eval(content.split("=", 1)[1].strip())

# print("Samples:", len(data))
# print(data[:5])

In [121]:
pairs = []

with open("english_telugu_sentences.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines()]

i = 0

while i < len(lines):

    if lines[i].startswith("English:"):

        english = lines[i].replace("English:", "").strip()

        if i + 1 < len(lines) and lines[i+1].startswith("Telugu:"):

            telugu = lines[i+1].replace("Telugu:", "").strip()

            pairs.append((english, telugu))

    i += 1

print("Total pairs:", len(pairs))

print(pairs[:5])

Total pairs: 28500
[('I go to school daily.', 'nenu roju school ku veltanu.'), ('I go to the office daily.', 'nenu roju office ku veltanu.'), ('I go to the market daily.', 'nenu roju market ku veltanu.'), ('I go to the hospital daily.', 'nenu roju hospital ku veltanu.'), ('I go to college daily.', 'nenu roju college ku veltanu.')]


In [122]:
import pandas as pd

df = pd.DataFrame(data, columns=["english", "telugu"])

df = df.drop_duplicates()

print("Unique samples:", len(df))

df.head()

Unique samples: 60


,english,telugu
0,i am a student,nenu vidyarthini
1,i am coming,nenu vasthunnanu
2,i am going,nenu velthunnanu
3,i need water,naaku neellu kavali
4,i like coffee,naaku coffee ishtam


In [123]:
df["telugu"] = df["telugu"].apply(
    lambda x: "<start> " + x + " <end>"
)

print(df.head())

          english                             telugu
0  i am a student     <start> nenu vidyarthini <end>
1     i am coming     <start> nenu vasthunnanu <end>
2      i am going     <start> nenu velthunnanu <end>
3    i need water  <start> naaku neellu kavali <end>
4   i like coffee  <start> naaku coffee ishtam <end>


In [124]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 48
Validation: 12


In [125]:
VOCAB_SIZE = 3000
SEQ_LEN = 7

source_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LEN
)

target_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LEN
)

source_vectorizer.adapt(train_df["english"])
target_vectorizer.adapt(train_df["telugu"])

print("English vocab size:",
      len(source_vectorizer.get_vocabulary()))

print("Telugu vocab size:",
      len(target_vectorizer.get_vocabulary()))

English vocab size: 34
Telugu vocab size: 37


In [126]:
# English → Integer tokens

train_source = source_vectorizer(
    train_df["english"].values
)

val_source = source_vectorizer(
    val_df["english"].values
)

# Telugu → Integer tokens

train_target = target_vectorizer(
    train_df["telugu"].values
)

val_target = target_vectorizer(
    val_df["telugu"].values
)

print(train_source.shape)
print(train_target.shape)

(48, 7)
(48, 7)


In [127]:
decoder_input_train = train_target[:, :-1]
decoder_target_train = train_target[:, 1:]

decoder_input_val = val_target[:, :-1]
decoder_target_val = val_target[:, 1:]

print(decoder_input_train.shape)
print(decoder_target_train.shape)

(48, 6)
(48, 6)


In [128]:
BATCH_SIZE = 12

train_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "encoder_inputs": train_source,
            "decoder_inputs": decoder_input_train
        },
        decoder_target_train
    )
)

train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "encoder_inputs": val_source,
            "decoder_inputs": decoder_input_val
        },
        decoder_target_val
    )
)

val_ds = val_ds.batch(BATCH_SIZE)

print("Dataset Ready")

Dataset Ready


In [129]:
class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(self, vocab_size, embed_dim, max_len):
        super().__init__()

        self.token_embeddings = Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embeddings = Embedding(
            max_len,
            embed_dim
        )

    def call(self, x):

        length = tf.shape(x)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embeddings(x)

        embedded_positions = self.position_embeddings(
            positions
        )

        return embedded_tokens + embedded_positions

In [130]:
EMBED_DIM = 64
LATENT_DIM = 128
NUM_HEADS = 2

SOURCE_VOCAB = len(
    source_vectorizer.get_vocabulary()
)

TARGET_VOCAB = len(
    target_vectorizer.get_vocabulary()
)

print(SOURCE_VOCAB)
print(TARGET_VOCAB)

34
37


In [131]:
class TransformerEncoder(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.dense_proj = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm_1 = LayerNormalization()
        self.layernorm_2 = LayerNormalization()

    def call(self, inputs):

        attention_output = self.attention(
            inputs,
            inputs
        )

        proj_input = self.layernorm_1(
            inputs + attention_output
        )

        proj_output = self.dense_proj(
            proj_input
        )

        return self.layernorm_2(
            proj_input + proj_output
        )

In [132]:
class TransformerDecoder(tf.keras.layers.Layer):

    def __init__(self, embed_dim, latent_dim, num_heads):
        super().__init__()

        self.attention_1 = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.attention_2 = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.dense_proj = tf.keras.Sequential([
            Dense(latent_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm_1 = LayerNormalization()
        self.layernorm_2 = LayerNormalization()
        self.layernorm_3 = LayerNormalization()

    def get_causal_attention_mask(self, inputs):

        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]

        i = tf.range(seq_len)[:, None]
        j = tf.range(seq_len)

        mask = tf.cast(i >= j, dtype="int32")

        mask = tf.reshape(
            mask,
            (1, seq_len, seq_len)
        )

        mult = tf.concat(
            [[batch_size], [1], [1]],
            axis=0
        )

        return tf.tile(mask, mult)

    def call(self, inputs, encoder_outputs):

        causal_mask = self.get_causal_attention_mask(
            inputs
        )

        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask
        )

        out_1 = self.layernorm_1(
            inputs + attention_output_1
        )

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs
        )

        out_2 = self.layernorm_2(
            out_1 + attention_output_2
        )

        proj_output = self.dense_proj(
            out_2
        )

        return self.layernorm_3(
            out_2 + proj_output
        )

In [133]:
encoder_inputs = Input(
    shape=(SEQ_LEN,),
    dtype="int64",
    name="encoder_inputs"
)

x = PositionalEmbedding(
    SOURCE_VOCAB,
    EMBED_DIM,
    SEQ_LEN
)(encoder_inputs)

encoder_outputs = TransformerEncoder(
    EMBED_DIM,
    LATENT_DIM,
    NUM_HEADS
)(x)

In [134]:
decoder_inputs = Input(
    shape=(SEQ_LEN - 1,),
    dtype="int64",
    name="decoder_inputs"
)

x = PositionalEmbedding(
    TARGET_VOCAB,
    EMBED_DIM,
    SEQ_LEN - 1
)(decoder_inputs)

x = TransformerDecoder(
    EMBED_DIM,
    LATENT_DIM,
    NUM_HEADS
)(
    x,
    encoder_outputs
)

In [135]:
decoder_outputs = Dense(
    TARGET_VOCAB,
    activation="softmax"
)(x)

transformer = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

transformer.summary()

Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 7, 64)     │      2,624 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 6, 64)     │      2,752 │ decoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 7, 64)     │     50,048 │ positional_embed… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, 6, 64)     │     83,392 │ positional_embed… │
│ (TransformerDecode… │                   │            │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 6, 37)     │      2,405 │ transformer_deco… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 141,221 (551.64 KB)

 Trainable params: 141,221 (551.64 KB)

 Non-trainable params: 0 (0.00 B)

In [136]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model Compiled Successfully")

Model Compiled Successfully


In [137]:
EPOCHS = 50

history = transformer.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 128ms/step - accuracy: 0.3611 - loss: 2.8497 - val_accuracy: 0.5000 - val_loss: 1.9898
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5521 - loss: 1.7624 - val_accuracy: 0.5972 - val_loss: 1.6278
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5938 - loss: 1.4124 - val_accuracy: 0.5556 - val_loss: 1.5068
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6562 - loss: 1.1921 - val_accuracy: 0.6250 - val_loss: 1.4001
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7188 - loss: 1.0379 - val_accuracy: 0.6667 - val_loss: 1.3017
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7743 - loss: 0.8871 - val_accuracy: 0.7222 - val_loss: 1.2052
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8194 - loss: 0.7390 - val_accuracy: 0.7639 - val_loss: 1.1170
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8576 - loss: 0.6164 - val_accuracy: 0.7639 - val_loss: 1.0362

In [138]:
transformer.save("models/translator_model.keras")

print("Model Saved")

Model Saved


In [139]:
target_vocab = target_vectorizer.get_vocabulary()

target_index_lookup = dict(
    zip(
        range(len(target_vocab)),
        target_vocab
    )
)

print("Vocabulary Loaded")

Vocabulary Loaded


In [140]:
def translate(sentence):

    tokenized_input = source_vectorizer(
        [sentence]
    )

    decoded_sentence =      "<start>"

    for i in range(SEQ_LEN - 1):

        tokenized_target = target_vectorizer(
            [decoded_sentence]
        )[:, :-1]

        predictions = transformer.predict(
            {
                "encoder_inputs": tokenized_input,
                "decoder_inputs": tokenized_target
            },
            verbose=0
        )

        sampled_token_index = np.argmax(
            predictions[0, i, :]
        )

        sampled_token = target_index_lookup.get(
            sampled_token_index,
            ""
        )

        if sampled_token == "<end>":
            break

        decoded_sentence += " " + sampled_token

    return decoded_sentence

In [141]:
print(
    translate(
        "i am going to school"
    )
)

<start> nenu school ki velthunnanu end 


In [142]:
print(translate("i am coming"))
print(translate("he is going to market"))
print(translate("i need water"))
print(translate("good morning"))

<start> nenu vasthunnanu end   
<start> athanu market ki velthunnadu end 
<start> naaku neellu kavali end  
<start> subhodayam end    


In [145]:
print(translate("he went to market"))

<start> athanu market ki velthunnadu end 
